In [0]:
from pyspark.sql import functions as F
import uuid

source_path = (
    "/Volumes/customer360_dev/staging/"
    "source_files/customers/incremental/"
)

target_table = "customer360_dev.bronze.customers"

run_id = str(uuid.uuid4())

print("Run ID:", run_id)

Run ID: 1d960525-3814-4455-87ea-fbca792d2bd0


In [0]:
files = dbutils.fs.ls(source_path)

csv_files = [
    f.path
    for f in files
    if f.path.lower().endswith(".csv")
]

print("Incremental CSV files found:", len(csv_files))

for file_path in csv_files:
    print(file_path)

Incremental CSV files found: 0


In [0]:
from pyspark.sql.types import StructType, StructField, StringType

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("customer_status", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("preferred_language", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
])

incremental_df = (
    spark.read
    .option("header", True)
    .schema(customers_schema)
    .csv(source_path)
)

In [0]:
dbutils.fs.ls(source_path)

[]

In [0]:
bronze_incremental_df = (
    incremental_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_system",
        F.lit("CRM")
    )
    .withColumn(
        "_load_type",
        F.lit("INCREMENTAL")
    )
    .withColumn(
        "_pipeline_run_id",
        F.lit(run_id)
    )
)

In [0]:
display(
    bronze_incremental_df.limit(20)
)

customer_id,first_name,last_name,email,phone_number,date_of_birth,gender,signup_date,customer_status,loyalty_tier,preferred_language,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,_pipeline_run_id


Count Incoming Incremental Records

In [0]:
incoming_count = bronze_incremental_df.count()

print(
    f"Incoming customer incremental rows: "
    f"{incoming_count:,}"
)

Incoming customer incremental rows: 0


In [0]:
# ============================================================
# 7. Append Incremental Records to Bronze
# ============================================================

if incoming_count > 0:

    (
        bronze_incremental_df
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(
            "customer360_dev.bronze.customers"
        )
    )

    print("Bronze incremental load completed.")

else:

    print(
        "No incremental customer records found. "
        "Bronze write skipped."
    )

No incremental customer records found. Bronze write skipped.


In [0]:
# ============================================================
# 8. Verify Current Pipeline Run
# ============================================================

if incoming_count > 0:

    written_count = (
        spark.table(
            "customer360_dev.bronze.customers"
        )
        .filter(
            F.col("_pipeline_run_id") == run_id
        )
        .count()
    )

else:
    written_count = 0

print(f"Incoming rows : {incoming_count:,}")
print(f"Written rows  : {written_count:,}")

Incoming rows : 0
Written rows  : 0


In [0]:
# ============================================================
# 9. Reconciliation Check
# ============================================================

difference = incoming_count - written_count

status = (
    "PASS"
    if difference == 0
    else "FAIL"
)

print(f"Difference : {difference:,}")
print(f"Status     : {status}")

Difference : 0
Status     : PASS


In [0]:
# ============================================================
# 10. Verify Load Type
# ============================================================

if incoming_count > 0:

    (
        spark.table(
            "customer360_dev.bronze.customers"
        )
        .filter(
            F.col("_pipeline_run_id") == run_id
        )
        .groupBy("_load_type")
        .count()
        .show()
    )

else:

    print(
        "Load type validation skipped - "
        "no incremental rows."
    )

Load type validation skipped - no incremental rows.


In [0]:
# ============================================================
# 11. Verify Source File Traceability
# ============================================================

if incoming_count > 0:

    (
        spark.table(
            "customer360_dev.bronze.customers"
        )
        .filter(
            F.col("_pipeline_run_id") == run_id
        )
        .groupBy("_source_file")
        .count()
        .show(truncate=False)
    )

else:

    print(
        "Source file validation skipped - "
        "no incremental files were processed."
    )

Source file validation skipped - no incremental files were processed.
